# Day 14: Mini Transformer 验收与计时

先验证同权重的 Cache/no-cache 逐 token logits，再计时。warm-up 在计时外，GPU 前后同步；报告 shape、dtype、重复次数、峰值 allocated memory。

参考源码：`learning/mini_transformer/model.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY08_14.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 14完整讲义](../docs/DAY14_MODEL_BASELINE.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

把Benchmark理解成同一条跑道上的计时，不是一个人跑100米、另一个跑10米：

```text
同权重 + 同输入 + 同输出预算 + 同dtype/设备
     ↓
full/cached与因果性先对齐
     ↓
预热 → 同步 → 重复采样 → 中位数/范围
```

prompt长8、输出4时，当前generate先prefill 8位置，
再把前三个输出逐个送回，最后一个输出不再forward。
因此缓存最多11位置，不是12；计时覆盖的是这整次生成，而非单token耗时。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(14, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

若 JIT 和首次分配包含在计时中会怎样？一次短 prompt 能证明长期吞吐吗？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：一次生成4个token的ms可以直接当TPOT吗？**
答：不能，含prefill、首token及多个decode，应按明确定义拆阶段或记录token时间戳。

**问：不同prompt长度比较谁更快合理吗？**
答：可作为不同场景报告，不能作为同工作负载的优化对照。

**问：为何不用一次最短耗时作为结论？**
答：会受缓存、调度、噪声影响，应预热、多次测量并报告统计口径。

**问：本周通过意味着什么？**
答：能独立说明并验证模型核心、Cache和梯度，不代表随机模型有语言能力或生产性能。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY14_MODEL_BASELINE.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。